# BloodMNIST: CNN on synthetic images

Part of an ongoing MedMNIST image-generation and classification project. See the repository README for progress and setup.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install numpy matplotlib


In [ ]:
import numpy as np
import os

# Define the correct path based on Kaggle storage
data_path = "/kaggle/input/kaun-hu-mai/is_bar_chal_ja.npz"  # Change this path if needed

# Load the dataset
def load_custom_dataset(file_path):
    print(f"Loading dataset from: {file_path}")

    data = np.load(file_path)

    # Extract data
    train_images = data["train_images"]
    train_labels = data["train_labels"]
    val_images = data["val_images"]
    val_labels = data["val_labels"]
    test_images = data["test_images"]
    test_labels = data["test_labels"]

    print("\nDataset Summary:")
    print(f"Train Images: {train_images.shape}, Labels: {train_labels.shape}")
    print(f"Validation Images: {val_images.shape}, Labels: {val_labels.shape}")
    print(f"Test Images: {test_images.shape}, Labels: {test_labels.shape}")

    return train_images, train_labels, val_images, val_labels, test_images, test_labels

train_images, train_labels, val_images, val_labels, test_images, test_labels = load_custom_dataset(data_path)


In [ ]:
class_labels = ['Class 0',
                'Class 1',
                'Class 2',
                'Class 3',
                'Class 4',
                'Class 5',
                'Class 6',
                'Class 7']

# Preprocessing

In [ ]:
# Define expected dataset sizes
expected_train = 11959
expected_val = 1712
expected_test = 3421

# Check missing images in the dataset
if train_images.shape[0] != expected_train:
    print("WARNING: Missing images in training dataset.")
    print(f"Found {train_images.shape[0]}, should be {expected_train}.")
else:
    print("SUCCESS: No missing images in training dataset.")

if val_images.shape[0] != expected_val:
    print("WARNING: Missing images in validation dataset.")
    print(f"Found {val_images.shape[0]}, should be {expected_val}.")
else:
    print("SUCCESS: No missing images in validation dataset.")

if test_images.shape[0] != expected_test:
    print("WARNING: Missing images in testing dataset.")
    print(f"Found {test_images.shape[0]}, should be {expected_test}.")
else:
    print("SUCCESS: No missing images in test dataset.")


In [ ]:
# Normalising images in the dataset
# Pixel values lie between 0-255,
# division by 255 will scale the values between 0 and 1
train_dataset.imgs = train_dataset.imgs/255.0
val_dataset.imgs = val_dataset.imgs/255.0
test_dataset.imgs = test_dataset.imgs/255.0

In [ ]:
# Normalize the images (0-1 scaling)
train_images = train_images.astype("float32") / 255.0
val_images = val_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0

# Ensure labels are integers
train_labels = train_labels.astype("int")
val_labels = val_labels.astype("int")
test_labels = test_labels.astype("int")
# Ensure grayscale images have an extra channel
if train_images.ndim == 3:
    print("Adding channel to images...")
    train_images = np.expand_dims(train_images, axis=-1)
    val_images = np.expand_dims(val_images, axis=-1)
    test_images = np.expand_dims(test_images, axis=-1)

print(f"Processed Train Images Shape: {train_images.shape}")
print(f"Processed Validation Images Shape: {val_images.shape}")
print(f"Processed Test Images Shape: {test_images.shape}")


In [ ]:
import matplotlib.pyplot as plt

def show_sample_images(images, labels, num_samples=5):
    plt.figure(figsize=(10, 2))
    for i in range(num_samples):
        plt.subplot(1, num_samples, i + 1)
        plt.imshow(images[i], cmap="gray")
        plt.title(f"Label: {labels[i][0]}")
        plt.axis("off")
    plt.show()

# Display sample training images
show_sample_images(train_images, train_labels)


# Model portion

In [ ]:
from pathlib import Path
from keras.models import model_from_json

def save_model(model, model_name):
    """Saves the model architecture in JSON format and weights in an H5 file."""
    save_path = Path(f"/kaggle/working/{model_name}")  # Kaggle directory
    save_path.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

    # Save model structure
    model_structure = model.to_json()
    (save_path / f"{model_name}.json").write_text(model_structure)

    # Save model weights (NOW with correct naming)
    model.save_weights(str(save_path / f"{model_name}.weights.h5"))

    print(f"✅ Model saved successfully in {save_path}")

def load_model(model_name, compile_model=True):
    """Loads the model architecture and weights from Kaggle's working directory."""
    load_path = Path(f"/kaggle/working/{model_name}")

    # Load model structure
    model_structure = (load_path / f"{model_name}.json").read_text()
    model = model_from_json(model_structure)

    # Load model weights (NOW with correct naming)
    model.load_weights(str(load_path / f"{model_name}.weights.h5"))

    # Compile the model if needed
    if compile_model:
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    print(f"✅ Model loaded successfully from {load_path}")
    return model


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Define a basic model (Modify as needed)
model = Sequential([
    Dense(64, activation='relu', input_shape=(10,)),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("✅ Model has been defined.")


In [ ]:
save_model(model, "my_model")
print("✅ Model saved successfully.")


In [ ]:
loaded_model = load_model("my_model")
print("✅ Model loaded successfully.")


# Model Evaluation

In [ ]:
# Function for Model Evaluation

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

def evaluate_model(true_labels, predicted_labels, predicted_probs, label_names):
  if(true_labels.ndim==2):
    true_labels = true_labels[:, 0]
  if(predicted_labels.ndim==2):
    predicted_labels = predicted_labels[:, 0]
  if(predicted_probs.ndim==2):
    predicted_probs = predicted_probs[:, 0]

  print(f"Accuracy : {accuracy_score(true_labels, predicted_labels)}")

  print(f"Precision : {precision_score(true_labels, predicted_labels,average='micro')}")

  print(f"Recall : {recall_score(true_labels, predicted_labels,average='micro')}")

  print(f"F1 Score: {f1_score(true_labels, predicted_labels,average='micro')}")

  # print(f"ROC AUC Score : {roc_auc_score(true_labels, predicted_probs)}")

  print("Classification report : ")
  print(classification_report(true_labels, predicted_labels, target_names=label_names))

  matrix = confusion_matrix(true_labels, predicted_labels)
  print("Confusion Matrix : ")

  ConfusionMatrixDisplay(matrix, display_labels=label_names).plot(cmap=plt.cm.Blues)
  if label_names is not None:
        tick_marks = np.arange(len(label_names))
        plt.xticks(tick_marks, label_names, rotation=45)
        plt.yticks(tick_marks, label_names)

 # # ---- ROC Curve Section ----
 #  true_labels_binarized = label_binarize(true_labels, classes=np.arange(len(label_names)))

 #  plt.figure(figsize=(10, 15))
 #  for i in range(len(label_names)):
 #      fpr, tpr, _ = roc_curve(true_labels_binarized[:, i], predicted_probs[:, i])
 #      roc_auc = auc(fpr, tpr)
 #      plt.plot(fpr, tpr, label=f"{label_names[i]} (AUC = {roc_auc:.2f})")

 #  plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line
 #  plt.xlim([0.0, 1.0])
 #  plt.ylim([0.0, 1.05])
 #  plt.xlabel('False Positive Rate')
 #  plt.ylabel('True Positive Rate')
 #  plt.title('Multi-Class ROC Curve')
 #  plt.legend(loc="lower right")
 #  plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

def evaluate_model(true_labels, predicted_labels, predicted_probs, label_names):
    # Check if the labels are one-hot encoded and convert them if necessary
    if(true_labels.ndim == 2):
        true_labels = true_labels[:, 0]
    if(predicted_labels.ndim == 2):
        predicted_labels = predicted_labels[:, 0]
    if(predicted_probs.ndim == 2):
        predicted_probs = predicted_probs[:, 0]

    # Print evaluation metrics
    print(f"Accuracy : {accuracy_score(true_labels, predicted_labels)}")
    print(f"Precision : {precision_score(true_labels, predicted_labels, average='micro')}")
    print(f"Recall : {recall_score(true_labels, predicted_labels, average='micro')}")
    print(f"F1 Score: {f1_score(true_labels, predicted_labels, average='micro')}")
    print("Classification report : ")
    print(classification_report(true_labels, predicted_labels, target_names=label_names))

    # Compute confusion matrix
    matrix = confusion_matrix(true_labels, predicted_labels)
    print("Confusion Matrix : ")

    # Create confusion matrix display with custom labels
    disp = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=label_names)

    # Increase resolution (DPI) and adjust font sizes for better visualization
    plt.figure(figsize=(10, 8), dpi=300)  # Set figure size and DPI for high resolution

    # Plot confusion matrix
    disp.plot(cmap=plt.cm.Blues)

    # Adjust font sizes for axes and labels
    plt.xticks(fontsize=12, rotation=45)  # Increase font size and rotate labels for clarity
    plt.yticks(fontsize=12)  # Increase font size for y-axis labels
    plt.xlabel('Predicted Labels', fontsize=14)  # Increase font size for x-axis label
    plt.ylabel('True Labels', fontsize=14)  # Increase font size for y-axis label
    plt.title('Confusion Matrix', fontsize=16)  # Set title font size

    # Adjust layout to prevent label clipping
    plt.tight_layout()

    # Save the plot as a high-resolution PNG image
    plt.savefig('confusion_matrix_high_res.png', dpi=300)

    # Show the plot
    plt.show()


# Visualization

import matplotlib.pyplot as plt

def plot_acc_loss(model_history):
    # Increase figure size and DPI for better resolution
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6), dpi=300)

    acc = model_history.history['accuracy']
    val_acc = model_history.history['val_accuracy']

    # Get number of epochs
    epochs = range(1, len(acc) + 1)

    # Plot training and validation accuracy per epoch
    ax1.plot(epochs, acc, label="Training acc")
    ax1.plot(epochs, val_acc, label="Validation acc")
    ax1.set_title('Training and validation accuracy', fontsize=18)
    ax1.set_xlabel('Epochs', fontsize=14)  # Increase font size
    ax1.set_ylabel('Accuracy', fontsize=14)  # Increase font size
    ax1.legend(fontsize=18)

    # Set xticks and yticks font size
    ax1.tick_params(axis='x', labelsize=16)
    ax1.tick_params(axis='y', labelsize=16)

    loss = model_history.history['loss']
    val_loss = model_history.history['val_loss']

    ax2.plot(epochs, loss, label="Training loss")
    ax2.plot(epochs, val_loss, label="Validation loss")
    ax2.set_title('Training and validation loss', fontsize=18)
    ax2.set_xlabel('Epochs', fontsize=14)  # Increase font size
    ax2.set_ylabel('Loss', fontsize=14)  # Increase font size
    ax2.legend(fontsize=18)

    # Set xticks and yticks font size
    ax2.tick_params(axis='x', labelsize=16)
    ax2.tick_params(axis='y', labelsize=16)

    # Adjust layout to prevent clipping of labels
    plt.tight_layout()

    # Save the figure with high resolution
    plt.savefig("accuracy_loss_plot_high_res.png", dpi=300)

    # Show the plot
    plt.show()


In [ ]:
# Function to plot Accuracy and Loss of a model

def plot_acc_loss(model_history):
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20,6), dpi=160)
  acc = model_history.history['accuracy']
  val_acc = model_history.history['val_accuracy']

  # Get number of epochs
  epochs = range(1, len(acc) + 1)

  # Plot training and validation accuracy per epoch
  ax1.plot(epochs, acc, label="Training acc")
  ax1.plot(epochs, val_acc, label="Validation acc")
  ax1.set_title('Training and validation accuracy')
  ax1.legend()

  loss = model_history.history['loss']
  val_loss = model_history.history['val_loss']

  ax2.plot(epochs, loss, label="Training loss")
  ax2.plot(epochs, val_loss, label="Validation loss")
  ax2.set_title('Training and validation loss')
  ax2.legend()

In [ ]:
import matplotlib.pyplot as plt

def plot_acc_loss(model_history):
    # Increase figure size and DPI for better resolution
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6), dpi=300)

    acc = model_history.history['accuracy']
    val_acc = model_history.history['val_accuracy']

    # Get number of epochs
    epochs = range(1, len(acc) + 1)

    # Plot training and validation accuracy per epoch
    ax1.plot(epochs, acc, label="Training acc")
    ax1.plot(epochs, val_acc, label="Validation acc")
    #ax1.set_title('Training and validation accuracy', fontsize=18)
    ax1.set_xlabel('Epochs', fontsize=14)  # Increase font size
    ax1.set_ylabel('Accuracy', fontsize=14)  # Increase font size
    ax1.legend(fontsize=18)

    # Set xticks and yticks font size
    ax1.tick_params(axis='x', labelsize=16)
    ax1.tick_params(axis='y', labelsize=16)

    loss = model_history.history['loss']
    val_loss = model_history.history['val_loss']

    ax2.plot(epochs, loss, label="Training loss")
    ax2.plot(epochs, val_loss, label="Validation loss")
    #ax2.set_title('Training and validation loss', fontsize=18)
    ax2.set_xlabel('Epochs', fontsize=14)  # Increase font size
    ax2.set_ylabel('Loss', fontsize=14)  # Increase font size
    ax2.legend(fontsize=18)

    # Set xticks and yticks font size
    ax2.tick_params(axis='x', labelsize=16)
    ax2.tick_params(axis='y', labelsize=16)

    # Adjust layout to prevent clipping of labels
    plt.tight_layout()

    # Save the figure with high resolution
    plt.savefig("accuracy_loss_plot_high_res.png", dpi=300)

    # Show the plot
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Function to visualize a subset of the dataset
def visualise_subset(images, labels):
    fig = plt.figure(figsize=(12, 10))
    for i in range(9):
        plt.subplot(330 + 1 + i)
        plt.imshow(images[i], cmap=plt.get_cmap('gray'))
        plt.title(f"Class {labels[i][0]}")
        plt.axis("off")  # Hide axis for cleaner visualization
    plt.show()

# Call the function for training images
visualise_subset(train_images, train_labels)


In [ ]:
import tensorflow as tf
from keras.utils import plot_model
from tensorflow import keras
from keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator # Changed import path
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

def augmented_data_visualisation(images, labels):
    datagen = ImageDataGenerator(rotation_range=30)
    train_generator = datagen.flow(images, labels, batch_size=1)  # Batch size = 1 to get one image at a time

    plt.figure(figsize=(10, 5))
    for i in range(6):
        plt.subplot(2, 3, i + 1)
        x, y = next(train_generator)  # Generate an augmented image
        plt.imshow(x[0], cmap='gray')  # Display the first image in the batch
        plt.title(f"Class {int(y[0])}")  # Convert label to integer
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Call the function for training images
augmented_data_visualisation(train_images, train_labels)


In [ ]:
from sklearn.utils import class_weight
import numpy as np

blood_class_weights = class_weight.compute_class_weight('balanced',
                                                         classes = np.unique(train_labels[:,0]),  # Modified
                                                         y = train_labels[:, 0])  # Modified

weights = {0 : blood_class_weights[0],
           1 : blood_class_weights[1],
           2 : blood_class_weights[2],
           3 : blood_class_weights[3],
           4 : blood_class_weights[4],
           5 : blood_class_weights[5],
           6 : blood_class_weights[6],
           7 : blood_class_weights[7] }

print(f"Class weights for imbalance {weights}")


# Data augmentation

In [ ]:
print("Preprocessing and augmentation of images with Standardisation, Rotation and Horizontal Flips..")
datagen = ImageDataGenerator(rotation_range=10,
                             horizontal_flip=True)

In [ ]:
my_callbacks = [tf.keras.callbacks.EarlyStopping(patience=10)]

In [ ]:
from tensorflow.keras.utils import to_categorical

# One-hot encoding of labels (multi-class classification)
train_labels = to_categorical(train_labels, num_classes=8)  # Modified
val_labels = to_categorical(val_labels, num_classes=8)  # Modified


In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D

model = Sequential()

model.add(Conv2D(32, (3, 3), padding='same', input_shape=(28, 28, 3), activation="relu"))
model.add(Conv2D(32, (3, 3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), padding='same', activation="relu"))
model.add(Conv2D(64, (3, 3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(512, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(8, activation="softmax"))

In [ ]:
# Printing model summary
print(model.summary())

In [ ]:
# Plotting model architecture
plot_model(model, to_file='model_from_scratch_blood.png', show_shapes=True, show_layer_activations=True)

In [ ]:
# Compiling the model
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
# Training the model
history = model.fit(datagen.flow(train_images, train_labels),  # Modified
                    epochs=100,
                    callbacks=my_callbacks,
                    validation_data=(val_images, val_labels),  # Modified
                    shuffle=True)


In [ ]:
save_model(model, "model_from_scratch_blood")

In [ ]:
# Predictions on test dataset
test_predict_prob = model.predict(test_images, verbose=0)  # Modified
test_predict_labels = np.argmax(test_predict_prob, axis=-1)


In [ ]:
# Model evaluation
evaluate_model(test_labels, test_predict_labels, test_predict_prob,class_labels)  # Modified


In [ ]:
# Plotting accuracy and loss
plot_acc_loss(history)

In [ ]:
print(weights)


In [ ]:
import numpy as np
unique, counts = np.unique(train_labels.argmax(axis=1), return_counts=True)
print("Class Distribution:", dict(zip(unique, counts)))


In [ ]:
unique_preds, pred_counts = np.unique(test_predict_labels, return_counts=True)
print("Predictions Distribution:", dict(zip(unique_preds, pred_counts)))
